# 03 — Feature engineering & encodage

**Rôle de ce notebook** : partir des tables maîtres (`train_master.csv` / `test_master.csv`) et
construire les matrices `X_train` / `X_test` / `y_train` pour la modélisation.

Choix principaux :

- **Cible : `log1p(Tonnage (Kg))`**. `Surface` est fournie comme feature, ce qui évite de diviser
  la cible par une surface parfois imputée.
- **Target encoding out-of-fold** (K-Fold, implémenté manuellement) des variables catégorielles :
  chaque ligne de train reçoit une moyenne de catégorie calculée sans elle-même, et le test reçoit
  la moyenne calculée sur tout le train.

In [1]:

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA_DIR = Path(r"C:\Github_Projects\Competition_Kaggle\Kaggle_competition\Data")
OUT_DIR = DATA_DIR / "processed"

train = pd.read_csv(OUT_DIR / "train_master.csv")
test = pd.read_csv(OUT_DIR / "test_master.csv")
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
print(train.shape, test.shape)


(1629, 61) (411, 60)


## 1. Features dérivées de l'âge

Le guide décrit une courbe âge-rendement sigmoïdale. On fournit à la fois l'âge brut et ses puissances (âge², âge³) pour que les modèles à base d'arbres puissent capturer la non-linéarité sans imposer de forme paramétrique.

In [2]:

for df in (train, test):
    df["Age_recolte_sq"] = df["Age_recolte"] ** 2
    df["Age_recolte_cb"] = df["Age_recolte"] ** 3

train[["Age_recolte", "Age_recolte_sq", "Age_recolte_cb"]].describe()


,Age_recolte,Age_recolte_sq,Age_recolte_cb
count,1625.000000,1625.000000,1625.000000
mean,17.570827,621.301461,32944.667737
std,17.685021,1306.553983,95475.835520
min,1.519507,2.308902,3.508393
25%,5.412731,29.297657,158.580336
50%,13.423682,180.195249,2418.883800
75%,18.666667,348.444444,6504.296296
max,81.248460,6601.312246,536346.453671


## 2. Ratios agronomiques

Ratio N/K, apports NPK par unité de surface, irrigation par unité de surface, efficience hydrique (guide §1.4.1).

In [3]:

eps = 1e-6
for df in (train, test):
    df["ratio_N_K_12m"] = df["N_cum_12m"] / (df["K_cum_12m"] + eps)
    df["npk_total_12m"] = df["N_cum_12m"] + df["P_cum_12m"] + df["K_cum_12m"]
    df["npk_par_surface_12m"] = df["npk_total_12m"] / (df["Surface"] + eps)
    df["irrigation_par_surface_12m"] = df["Irrigation_m3_cum_12m"] / (df["Surface"] + eps)
    df["efficience_hydrique_N-1"] = df["rendement_N-1"] / (df["irrigation_par_surface_12m"] * df["Surface"] + eps)

train[["ratio_N_K_12m", "npk_par_surface_12m", "irrigation_par_surface_12m"]].describe()


,ratio_N_K_12m,npk_par_surface_12m,irrigation_par_surface_12m
count,1.583000e+03,1583.000000,1583.000000
mean,1.441756e+06,59.678534,3104.450297
std,1.091585e+07,91.910699,2878.539217
min,0.000000e+00,0.000000,0.000000
25%,2.401695e-03,0.932917,1319.224647
50%,9.650807e-01,22.950904,2351.967904
75%,2.224809e+00,80.353469,3804.003379
max,1.809937e+08,1391.347875,33374.972638


## 3. Encodage cyclique du mois et saison agricole

In [4]:

def add_month_features(df):
    df = df.copy()
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

train = add_month_features(train)
test = add_month_features(test)


## 4. Target Encoding out-of-fold (sans fuite de données)

Pour chaque colonne catégorielle, la valeur encodée d'une ligne de train est calculée à partir des **autres** folds uniquement (K=5), avec un lissage vers la moyenne globale pour les catégories rares. Le test est encodé avec la moyenne calculée sur l'intégralité du train.

In [5]:

CAT_COLS = ["Region", "Libelle Variete", "Porte Greffe", "Libelle Site", "Code Ferme"]
TARGET = "Tonnage (Kg)"
SMOOTHING = 10

global_mean = np.log1p(train[TARGET]).mean()

def smoothed_means(frame, col, target_log, smoothing=SMOOTHING):
    stats = frame.groupby(col)[target_log].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    return smoothed

train["_log_target"] = np.log1p(train[TARGET])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for col in CAT_COLS:
    train[f"{col}_te"] = np.nan
    for train_idx, val_idx in kf.split(train):
        means = smoothed_means(train.iloc[train_idx], col, "_log_target")
        train.loc[train.index[val_idx], f"{col}_te"] = (
            train.iloc[val_idx][col].map(means).fillna(global_mean).values
        )
    # encodage final (fit sur tout train) pour le test
    full_means = smoothed_means(train, col, "_log_target")
    test[f"{col}_te"] = test[col].map(full_means).fillna(global_mean)

train = train.drop(columns=["_log_target"])
print("Colonnes encodées :", [f"{c}_te" for c in CAT_COLS])
train[[f"{c}_te" for c in CAT_COLS]].describe()


Colonnes encodées : ['Region_te', 'Libelle Variete_te', 'Porte Greffe_te', 'Libelle Site_te', 'Code Ferme_te']


,Region_te,Libelle Variete_te,Porte Greffe_te,Libelle Site_te,Code Ferme_te
count,1629.000000,1629.000000,1629.000000,1629.000000,1629.000000
mean,9.165400,9.265674,9.157921,9.123972,9.123972
std,0.950791,1.138273,0.994664,1.232664,1.232664
min,7.730928,4.490257,7.241964,6.513001,6.513001
25%,7.865053,8.847764,9.103620,8.772750,8.772750
50%,9.279138,9.532053,9.618183,9.379724,9.379724
75%,10.225660,10.018112,9.727512,9.884520,9.884520
max,10.425503,10.784221,10.088029,10.905918,10.905918


## 5. Gestion du cold-start (parcelles de test absentes de train)

Les 13 `CodeTracabilite` présents en test mais absents de train reçoivent l'encodage par défaut (`global_mean`, déjà géré par le `.fillna(global_mean)` ci-dessus) — c'est le repli hiérarchique le plus simple. On matérialise un flag pour pouvoir suivre séparément la performance du modèle sur ces lignes en validation (§ notebook 04).

In [6]:

train_codes = set(train["CodeTracabilite"])
test["cold_start"] = (~test["CodeTracabilite"].isin(train_codes)).astype(int)
print(f"Lignes test en cold-start (parcelle jamais vue en train) : {test['cold_start'].sum()} / {len(test)}")


Lignes test en cold-start (parcelle jamais vue en train) : 18 / 411


## 6. Assemblage de X_train / X_test / y_train

In [7]:

FEATURES = [
    "Surface", "Age_recolte", "Age_recolte_sq", "Age_recolte_cb",
    "month_sin", "month_cos",
    "Irrigation_m3_cum_1m", "Irrigation_m3_cum_3m", "Irrigation_m3_cum_6m", "Irrigation_m3_cum_12m",
    "N_cum_1m", "N_cum_3m", "N_cum_6m", "N_cum_12m",
    "P_cum_1m", "P_cum_3m", "P_cum_6m", "P_cum_12m",
    "K_cum_1m", "K_cum_3m", "K_cum_6m", "K_cum_12m",
    "ratio_N_K_12m", "npk_total_12m", "npk_par_surface_12m", "irrigation_par_surface_12m",
    "TemperatureMoy_C", "TemperatureMax_C", "TemperatureMin_C",
    "HumiditeRelativeMoy_pct", "ET0_mm",
    "TemperatureMoy_C_roll3m", "ET0_mm_roll3m", "TemperatureMoy_C_roll6m", "ET0_mm_roll6m",
    "rendement_N-1", "rendement_N-1_manquant",
    "parcelle_info_manquante", "age_incoherente",
    "Region_te", "Libelle Variete_te", "Porte Greffe_te", "Libelle Site_te", "Code Ferme_te",
]
FEATURES = [c for c in FEATURES if c in train.columns and c in test.columns]

META_COLS = ["id", "CodeTracabilite", "Campagne", "Date", "Surface", "Tonnage (Kg)"]

X_train = train[FEATURES].copy()
y_train = np.log1p(train["Tonnage (Kg)"])
X_test = test[FEATURES].copy()
meta_train = train[[c for c in META_COLS if c in train.columns]].copy()
meta_test = test[[c for c in ["id", "CodeTracabilite", "Campagne", "Date", "Surface", "cold_start"] if c in test.columns]].copy()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}")
print(f"\nFeatures manquantes (fraction) dans X_train:")
(X_train.isna().mean().sort_values(ascending=False).head(10))


X_train: (1629, 44), X_test: (411, 44), y_train: (1629,)

Features manquantes (fraction) dans X_train:


rendement_N-1            0.311234
Irrigation_m3_cum_6m     0.028238
Irrigation_m3_cum_1m     0.028238
Irrigation_m3_cum_12m    0.028238
N_cum_1m                 0.028238
N_cum_3m                 0.028238
P_cum_12m                0.028238
P_cum_6m                 0.028238
N_cum_12m                0.028238
P_cum_1m                 0.028238
dtype: float64

In [8]:

X_train.to_csv(OUT_DIR / "X_train.csv", index=False)
X_test.to_csv(OUT_DIR / "X_test.csv", index=False)
y_train.to_csv(OUT_DIR / "y_train.csv", index=False)
meta_train.to_csv(OUT_DIR / "meta_train.csv", index=False)
meta_test.to_csv(OUT_DIR / "meta_test.csv", index=False)

print("Fichiers sauvegardés dans", OUT_DIR)
for f in ["X_train.csv", "X_test.csv", "y_train.csv", "meta_train.csv", "meta_test.csv"]:
    print(" -", f)


Fichiers sauvegardés dans C:\Github_Projects\Competition_Kaggle\Kaggle_competition\Data\processed
 - X_train.csv
 - X_test.csv
 - y_train.csv
 - meta_train.csv
 - meta_test.csv
